[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/58_vit_block_solution.ipynb)

# 🔴 Solution: Vision Transformer Block

Reference solution for a single ViT encoder block: pre-norm multi-head self-attention followed by a pre-norm MLP with GELU activation.

$$x \leftarrow x + \text{MHA}(\text{LN}_1(x)), \quad x \leftarrow x + \text{MLP}(\text{LN}_2(x))$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✅ SOLUTION

class ViTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_h = d_model // num_heads
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def _gelu(self, x):
        return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

    def _mha(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scale = self.d_h ** -0.5
        attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(out)

    def forward(self, x):
        x = x + self._mha(self.norm1(x))
        x = x + self.fc2(self._gelu(self.fc1(self.norm2(x))))
        return x

In [ ]:
torch.manual_seed(0)
batch, num_patches, d_model, num_heads = 2, 16, 64, 4

block = ViTBlock(d_model, num_heads)
x = torch.randn(batch, num_patches, d_model)
out = block(x)

print("Input shape: ", x.shape)    # (2, 16, 64)
print("Output shape:", out.shape)  # (2, 16, 64)
assert out.shape == x.shape, "Shape mismatch!"
print("Shape preserved: True")

In [ ]:
from torch_judge import check
check("vit_block")